In [ ]:
# 한기대 장학금 정보 수집하여 저장하기
print("=" *80)
print(" 한기대 장학금 - 저장할 내용을 목록으로 만들어서 xls , csv 형식으로 저장하기")
print("=" *80)

#Step 0. 필요한 모듈과 라이브러리를 로딩하고 검색어를 입력 받습니다
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.service import Service
from selenium.common.exceptions import UnexpectedAlertPresentException, NoAlertPresentException
from selenium.webdriver.common.alert import Alert
# 엑셀파일 실행하기
import win32com.client as win32   #pywin32 , pypiwin32 설치후 동작
import win32api  #파이썬 프롬프트를 관리자 권한으로 실행해야 에러없음
                 #파이썬 쉘을 관리자 권한으로 실행한 후 불러오기로 이 소스 실행하기
import time
import sys        # system 설정을 변경하기 위해 필요합니다
import math
import pandas  as pd   
import os
import openpyxl

query_txt = '장학금' #input('1.크롤링할 키워드는 무엇입니까?: ')
cnt=int(input('2.수집할 데이터는 몇 건입니까?: ') )
page_cnt = math.ceil(cnt / 10)

f_dir = input('3.결과를 저장할 폴더이름을 입력해주세요(기본경로: c:\\py_temp\\) :')
if f_dir =='' :
    f_dir = 'c:\\py_temp\\'
    
#Step 1. 크롬 드라이버를 사용해서 웹 브라우저를 실행합니다.
# import chromedriver_autoinstaller
# chromedriver_autoinstaller.install()
s = Service("c:/py_temp/chromedriver.exe")
driver = webdriver.Chrome(service=s)

driver.get('https://www.koreatech.ac.kr')
time.sleep(5)

#Step 2. 홈페이지에서 장학금 검색
element = driver.find_element(By.ID,'search2')
driver.find_element(By.XPATH,'//*[@id="search2"]').click( )
element.send_keys(query_txt)
element.send_keys("\n")
time.sleep(5)

#Step 6.장학금 클릭하기
driver.find_element(By.XPATH,'//*[@id="contents_body"]/div[2]/ul/li[4]/a').click()
time.sleep(5)

no2 = [ ]           # 게시글 번호 컬럼
title2 = [ ]        # 게시물 제목 컬럼
bdate2 = [ ]        # 작성 일자 컬럼
body2=  [ ]         # 본문 내용
no = 1

print("\n")
html = driver.page_source
soup = BeautifulSoup(html, 'html.parser')
for page_number in range(1, page_cnt + 1) :
    content_list = soup.find('div','search_view_box result_b').find_all('li')
    print("\n")
    print("%s 페이지 내용 수집 시작합니다 =======================" %page_number)
    for i in content_list :
        try :
            title = i.find('div','title').find('a').get_text()
        except :
            continue
        else :

            no2.append(no)                            # 게시물 번호 리스트에 추가
            print('1.번호:',no)
            title2.append(title)                      # 게시물 제목 리스트에 추가
            print('2.제목:',title)

            bdate = i.find('span','date').get_text()  # 작성일자
            bdate2.append(bdate)                     # 작성일자 리스트에 추가
            print('3.작성일자:',bdate)

            body = i.find('p','wisenut-content').get_text()
            body2.append(body)
            print('4.내용:',body)
        
            if no >= cnt :
                break
                
            no += 1
    driver.find_element(By.LINK_TEXT,'다음').click()


# 출력 결과를 표(데이터 프레임) 형태로 만들기

data = pd.DataFrame()
data['번호'] = no2
data['제목'] = title2
data['내용'] = body2
data['작성일자'] = bdate2

# 저장될 파일위치와 이름을 지정합니다
now = time.localtime()
s = '%04d-%02d-%02d-%02d-%02d-%02d' % (now.tm_year, now.tm_mon, now.tm_mday, now.tm_hour, now.tm_min, now.tm_sec)

sec_name = ' 한기대 장학금 정보'
os.makedirs(f_dir+s+'-'+query_txt+'-'+sec_name)
os.chdir(f_dir+s+'-'+query_txt+'-'+sec_name)

fc_name=f_dir+s+'-'+query_txt+'-'+sec_name+'\\'+s+'-'+query_txt+'-'+sec_name+'.csv'
fx_name=f_dir+s+'-'+query_txt+'-'+sec_name+'\\'+s+'-'+query_txt+'-'+sec_name+'.xlsx'

# csv 형태로 저장하기
data.to_csv(fc_name,encoding="utf-8-sig",index=False)
print(" csv 파일 저장 경로: %s" %fc_name) 

# 엑셀 형태로 저장하기
import openpyxl
data.to_excel(fx_name , index=False, engine='openpyxl')
print(" xls 파일 저장 경로: %s" %fx_name) 

# 엑셀파일 실행하기
import win32com.client as win32   #pywin32 , pypiwin32 설치후 동작
import win32api  #파이썬 프롬프트를 관리자 권한으로 실행해야 에러없음
                 #파이썬 쉘을 관리자 권한으로 실행한 후 불러오기로 이 소스 실행하기

excel = win32.Dispatch('Excel.Application')
wb = excel.Workbooks.Open(fx_name)
sheet = wb.ActiveSheet
excel.ActiveWorkbook.Save()
excel.Visible=True

driver.close( )

 한기대 장학금 - 저장할 내용을 목록으로 만들어서 xls , csv 형식으로 저장하기


KeyboardInterrupt: 